# Anomaly labels as evidence about synthetic mechanisms

This is forensic label analysis, not a fraud prediction exercise. UNLABELED denotes a blank source anomaly field and never means verified normal. All original labels, including CUSTOMER_RECALL, remain separate. Implementation: `anomaly_analysis`, `duplicate_features`, and `binary_comparison` in `src/remaining_analysis.py`.

All record-level caches remain under ignored outputs/cache. No accounts, token values, details, or transaction rows are exported in analytical tables. Observed-label membership may itself be an engineered generator branch. Intervals describe hypothetical independent generator draws, not causal or real-world uncertainty.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "data/cleaned/swift_payments_cleaned.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / ".venv/Scripts/python.exe").exists()
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve(), "Use the project virtual environment"
sys.path.insert(0, str(ROOT / "src"))
from remaining_analysis import Study, anomaly_analysis
study = Study(ROOT, '06_anomaly_analysis', prerequisites=('05_currency_party',))

## Label populations, amounts, routing, and adjusted duration

**Purpose.** Identify the scope and magnitude of label-specific differences.

**Hypothesis.** Anomaly categories have distinct amount or duration profiles that persist within comparable payment segments.

**Methodology.** Report every label with counts, medians and P95 in USD/seconds, route-party composition and Cramér V with BH p-values. Compare each label only against UNLABELED using exact route/participants/FX/currency/party strata, plus amount bands for duration. Repeat minimum arm counts of 5 and 20. Mean log(1+amount) and log(1+duration) comparisons are overlap weighted; analytic 95% intervals use independent-cell variances.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** Support and effect direction determine whether a candidate survives adjustment. A label difference can describe generator construction without diagnosing real fraud.

**Limitations.** Labels can be rare and non-independent across related payments. A small supported fraction limits the estimand to common strata. No causal mechanism is established.

**Business Implications.** Use label-specific profiles for synthetic-data QA and investigation design; do not operationalize them as fraud decisions.

In [ ]:
summaries, effects, recurrence, repeat_profiles = anomaly_analysis(study)
display(summaries)
display(effects)
display(Image(filename=str(study.charts / "anomaly_amount.png")))
display(Image(filename=str(study.charts / "anomaly_duration.png")))

## Duplicate labels and recurrence counterexamples

**Purpose.** Test whether duplicate-payment labels have observable relational evidence.

**Hypothesis.** DUPLICATE_PAYMENT rows more often share exact ordering account, beneficiary account, cleaned numeric amount serialization, and currency with another row, particularly within 86,400 seconds.

**Methodology.** Scan account identifiers read-only and validate internal grouping hashes against distinct exact tuples. Retain every transaction. Count recurrence anywhere in the population and any nearest same-signature neighbor within 86,400 seconds. Exclude blank-account signatures only from recurrence denominators, reporting evaluable counts. Contrast DUPLICATE_PAYMENT against UNLABELED with Newcombe 95% risk-difference intervals, risk ratios and continuity-corrected odds ratios. Amount-rounding frequency is a separate weak clue, not evidence of a corrected intended amount.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** A missing counterpart rejects the strict observable exact-signature hypothesis for that row. It cannot disprove a duplicate event outside the supplied window or with a changed field.

**Limitations.** Exact signature is deliberately strict; it misses near duplicates. The same amount/account pair can recur legitimately. Numeric serialization equality is narrower than tolerance-based amount matching. Time proximity is not evidence of intent.

**Business Implications.** Evaluate linkage coverage before relying on label semantics; retain both repeated and non-repeated counterexamples.

In [ ]:
display(repeat_profiles)
display(recurrence)
display(pd.read_csv(study.tables / "anomaly_amount_rounding.csv"))

In [ ]:
report = study.finish([f"{r.labeled}: n={r.n:,}, median amount USD {r.median_amount_usd:.6f}, median duration {r.median_duration_seconds:g} seconds." for r in summaries.itertuples()] +
                      [f"{r.feature}: duplicate-label rate {r.target_rate:.6%} (n={r.target_n:,}), unlabeled rate {r.baseline_rate:.6%} (n={r.baseline_n:,}); risk difference {r.risk_difference:.6f}, 95% CI [{r.risk_difference_ci_low:.6f}, {r.risk_difference_ci_high:.6f}]." for r in recurrence.itertuples()] +
                      ["An exact observable duplicate counterpart is a tested hypothesis, not an assumed label definition; inspect recurrence rates for exceptions."], ['Blank labels are not confirmed normal.', 'CUSTOMER_RECALL is undocumented and retained without reinterpretation.', 'Unobserved intended amounts and external counterpart transactions prevent definitive fat-finger or duplicate reconstruction.', 'Adjusted contrasts with weak support remain exploratory.'], 'Parse token field structure without publishing literal values, and test party/label associations.')
display(Markdown("### Final findings and limitations\n" + "\n".join("- " + x for x in report["findings"]) + "\n\n" + " ".join(report["limitations"])))
display(pd.DataFrame(study.checks))
display(Markdown("### Completion summary\nExecution passed for " + format(report["population"], ",") + " transactions. All generated files are listed below. No unresolved material errors.\n\n" + "\n".join("- `"+x+"`" for x in report["files"]) + "\n\nRecommended next investigation: " + report["next_investigation"]))